# Dogs vs Cats — Binary CNN

Trains a small convolutional binary classifier on the **cat** and **dog** subset of CIFAR-10. Uses 32×32 RGB images (10 000 cats + 10 000 dogs). Saved to `models/dogs_vs_cats.keras`.

## Part 0 — Setup

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print('TensorFlow', tf.__version__)

## Part 1 — Build a Cat / Dog dataset from CIFAR-10

CIFAR-10 has 10 classes; we keep only `cat` (label 3) and `dog` (label 5), then relabel:
- 0 → cat
- 1 → dog

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()

def keep_cats_dogs(X, y):
    mask = (y == 3) | (y == 5)
    X, y = X[mask], y[mask]
    y = (y == 5).astype('int32')   # cat=0, dog=1
    return X.astype('float32') / 255.0, y

X_train, y_train = keep_cats_dogs(X_train, y_train)
X_test,  y_test  = keep_cats_dogs(X_test,  y_test)
print('Train:', X_train.shape, '  Test:', X_test.shape)

### 1.1 Inspect the data

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
labels = ['cat', 'dog']
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(labels[int(y_train[i])])
    ax.axis('off')
plt.tight_layout(); plt.show()

## Part 2 — Build the CNN

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    augment,
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid'),  # binary
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Part 3 — Train

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=128,
    validation_data=(X_test, y_test),
    verbose=1,
)

### 3.1 Plot history

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(history.history['accuracy'], label='train')
ax[1].plot(history.history['val_accuracy'], label='val')
ax[1].set_title('Accuracy'); ax[1].legend()
plt.tight_layout(); plt.show()

## Part 4 — Evaluate

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {acc:.4f}')

## Part 5 — Save the Model

In [ ]:
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / 'dogs_vs_cats.keras'
model.save(MODEL_PATH)
print('Saved ->', MODEL_PATH, f'({MODEL_PATH.stat().st_size/1024:.1f} KB)')

## Part 6 — Predict Helper

In [ ]:
from PIL import Image

def predict_pet(img_or_path):
    if isinstance(img_or_path, (str, Path)):
        img = Image.open(img_or_path).convert('RGB')
    else:
        img = Image.fromarray(np.asarray(img_or_path)).convert('RGB')
    img = img.resize((32, 32))
    arr = np.asarray(img, dtype='float32') / 255.0
    p_dog = float(model.predict(np.expand_dims(arr, 0), verbose=0)[0][0])
    return ('dog' if p_dog > 0.5 else 'cat'), p_dog

label, p = predict_pet(X_test[0])
print(f'Sample 0 -> {label} (p_dog={p:.2%})')